Đây là mô hình Học Sâu (Deep Learning) sử dụng Word Embeddings và Mạng Tích chập 1 chiều (1D-CNN).

- Dữ liệu đầu vào: symptoms.csv
- Mục tiêu: Huấn luyện Mô hình 1 (Chính) để dự đoán outcome (0/1).

In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

In [6]:
# --- 1. Tải và chuẩn bị dữ liệu ---
try:
    df_nlp = pd.read_csv("../../datasets/raw/symptoms.csv")
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file 'symptoms.csv'. Vui lòng kiểm tra lại đường dẫn.")
    
df_nlp['text'] = df_nlp['text'].astype(str).fillna('')

# Dữ liệu cho Mô hình 1 (Chính): Dự đoán Outcome (0/1)
X = df_nlp['text']
y = df_nlp['outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Tổng số mẫu: {len(X)}")
print(f"Huấn luyện trên: {len(X_train)} mẫu")
print(f"Kiểm thử trên: {len(X_test)} mẫu")

Tổng số mẫu: 112
Huấn luyện trên: 89 mẫu
Kiểm thử trên: 23 mẫu


In [7]:
# --- 2. Vector hóa (Tokenizer & Padding) ---
# Thiết lập các tham số
VOCAB_SIZE = 5000  # Số lượng từ tối đa trong từ điển (từ vựng)
MAX_LEN = 50       # Độ dài tối đa của 1 câu (câu ngắn hơn sẽ được đệm)
EMBED_DIM = 100    # Số chiều của vector word embedding

# Tokenizer: Biến chữ thành số (index)
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>") # oov_token: từ không có trong từ điển
tokenizer.fit_on_texts(X_train) # Chỉ học từ vựng từ tập train

# Chuyển đổi văn bản thành chuỗi số
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences: Đảm bảo các câu có cùng độ dài (MAX_LEN)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

# Chuyển y_train, y_test thành numpy array (bắt buộc cho Keras)
y_train_np = np.array(y_train)
y_test_np = np.array(y_test)

In [8]:
# --- 3. Xây dựng mô hình 1D-CNN ---
print("\nĐang xây dựng mô hình 1D-CNN...")
model = Sequential([
    # 1. Lớp Embedding: Học cách biểu diễn vector cho từng từ
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN),
    
    Dropout(0.2),
    
    # 2. Lớp Conv1D: Bộ lọc (kernel_size=3) trượt qua câu để tìm cụm 3 từ
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    
    # 3. Lớp Pooling: Giữ lại đặc trưng quan trọng nhất
    GlobalMaxPooling1D(),
    
    # 4. Lớp Dense (Fully Connected)
    Dense(64, activation='relu'),
    Dropout(0.2),
    
    # 5. Lớp Output: Phân loại 0 hoặc 1
    Dense(1, activation='sigmoid') # Dùng sigmoid cho phân loại nhị phân (0/1)
])

# Compile mô hình
model.compile(loss='binary_crossentropy', # Dùng binary_crossentropy cho 0/1
              optimizer='adam', 
              metrics=['accuracy'])
model.summary()


Đang xây dựng mô hình 1D-CNN...


d:\IT\HK1_Y4\Class\Project_1\Project_2\.venv\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
# --- 4. Huấn luyện ---
print("\nĐang huấn luyện mô hình CNN...")
# (Với data 64 mẫu, 10-15 epochs là đủ. Nếu nhiều data hơn, hãy tăng lên)
history = model.fit(X_train_pad, y_train_np, 
                    epochs=15, 
                    validation_data=(X_test_pad, y_test_np),
                    batch_size=8, # Batch size nhỏ vì data ít
                    verbose=2) # verbose=2 để hiển thị ít log hơn


Đang huấn luyện mô hình CNN...
Epoch 1/15
12/12 - 2s - 202ms/step - accuracy: 0.5281 - loss: 0.6885 - val_accuracy: 0.6087 - val_loss: 0.6746
Epoch 2/15
12/12 - 0s - 19ms/step - accuracy: 0.5955 - loss: 0.6618 - val_accuracy: 0.6087 - val_loss: 0.6591
Epoch 3/15
12/12 - 0s - 17ms/step - accuracy: 0.5955 - loss: 0.6427 - val_accuracy: 0.6087 - val_loss: 0.6466
Epoch 4/15
12/12 - 0s - 17ms/step - accuracy: 0.6742 - loss: 0.6007 - val_accuracy: 0.6957 - val_loss: 0.6289
Epoch 5/15
12/12 - 0s - 19ms/step - accuracy: 0.8539 - loss: 0.5421 - val_accuracy: 0.7391 - val_loss: 0.5910
Epoch 6/15
12/12 - 0s - 17ms/step - accuracy: 0.9101 - loss: 0.4533 - val_accuracy: 0.6957 - val_loss: 0.5378
Epoch 7/15
12/12 - 0s - 20ms/step - accuracy: 0.9888 - loss: 0.3141 - val_accuracy: 0.6957 - val_loss: 0.4802
Epoch 8/15
12/12 - 0s - 18ms/step - accuracy: 0.9888 - loss: 0.2067 - val_accuracy: 0.7826 - val_loss: 0.4524
Epoch 9/15
12/12 - 0s - 17ms/step - accuracy: 0.9888 - loss: 0.1071 - val_accuracy: 0.7

In [10]:
# --- 5. Đánh giá ---
print("\n============================================================")
print("Kết quả đánh giá mô hình 1D-CNN (Mô hình 1 - Nhị phân)")
print("============================================================")
y_pred_proba = model.predict(X_test_pad)
y_pred = (y_pred_proba > 0.5).astype(int) # Chuyển xác suất (ví dụ 0.8) về nhãn (1)

print(f"Accuracy: {accuracy_score(y_test_np, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_np, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_np, y_pred, target_names=['0 (Không bị)', '1 (Bị)']))


Kết quả đánh giá mô hình 1D-CNN (Mô hình 1 - Nhị phân)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
Accuracy: 0.7826

Confusion Matrix:
[[ 7  2]
 [ 3 11]]

Classification Report:
              precision    recall  f1-score   support

0 (Không bị)       0.70      0.78      0.74         9
      1 (Bị)       0.85      0.79      0.81        14

    accuracy                           0.78        23
   macro avg       0.77      0.78      0.78        23
weighted avg       0.79      0.78      0.78        23



In [11]:
# --- 6. Lưu model và tokenizer ---
# model.save("../../models/nlp_cnn_binary_model.h5")
# joblib.dump(tokenizer, "../../models/nlp_binary_tokenizer.pkl")
# joblib.dump(MAX_LEN, "../../models/nlp_binary_max_len.pkl")
print("\n✅ Mô hình CNN (model_binary) và Tokenizer đã được lưu thành công!")


✅ Mô hình CNN (model_binary) và Tokenizer đã được lưu thành công!
